# CloudSplaining AWS Managed Policies (camp)

Code related to the first blog post: https://www.tenchisecurity.com/blog/aws-managed-iam-policies

First, we load the policy versions data with some slight transformations.

In [ ]:
import pandas
import datetime
import matplotlib

pd = pandas.read_csv('data/versions_summary.csv')
pd['CreateDate'] = pd['CreateDate'].apply(lambda x: pandas.Timestamp(datetime.datetime.fromisoformat(x)))
pd['VersionId'] = pd['VersionId'].apply(lambda x: int(x[1:].strip()))
pd.rename(columns={'VersionId': 'Version'}, inplace=True)
pd.info()

Let's try to get a sense of the AWS new policy publishing cadence, by measuring how many `v1` policies are published by quarter over time.

In [ ]:
# policies over time panda
pot = pd[(pd['Version'] == 1)][['CreateDate', 'PolicyName']]
pot.rename(columns={'CreateDate': 'Creation', 'PolicyName': 'Policies'}, inplace=True)
pot = pot.groupby(pandas.Grouper(key='Creation', freq="Q")).size()
pot.drop(pot.index[-1], inplace=True) # delete current quarter
pl = pot.plot(kind='bar', title='Number of New Policies Created per Quarter')
pl.xaxis.set_major_formatter(lambda x, pos: f"{pot.index[x].year}  Q{pot.index[x].quarter}")
pl.xaxis.get_label().set_visible(False)
pl

Now the cadence of new versions published over time:

In [ ]:
# policies over time panda
pot = pd[['CreateDate', 'PolicyName']]
pot.rename(columns={'CreateDate': 'Creation', 'PolicyName': 'Policies'}, inplace=True)
pot = pot.groupby(pandas.Grouper(key='Creation', freq="Q")).size()
pot.drop(pot.index[-1], inplace=True) # delete current quarter
pl = pot.plot(kind='bar', title='Number of New Policy Versions Created per Quarter')
pl.xaxis.set_major_formatter(lambda x, pos: f"{pot.index[x].year}  Q{pot.index[x].quarter}")
pl.xaxis.get_label().set_visible(False)
pl

Just out of curiosity, let's the breakdown of new version releases by month:

In [ ]:
# policies over time panda
pot = pd[['CreateDate']]
pot.rename(columns={'CreateDate': 'Published'}, inplace=True)
pot['Published'] = pot['Published'].apply(lambda x: x.strftime('%b'))
pot = pot.groupby(['Published'], as_index=False).size()
total_versions = sum(pot['size'])
pot = pot.set_index('Published')
pot = pot.reindex(reversed(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']))
pl = pot.plot(kind='barh', title='Policy Versions Published per Month')
pl.get_legend().remove()
pl.spines['top'].set_visible(False)
pl.spines['right'].set_visible(False)
pl.spines['bottom'].set_visible(False)
pl.spines['left'].set_visible(False)
pl.xaxis.set_visible(False)
pl.yaxis.get_label().set_visible(False)
for index,data in enumerate(pot['size']):
    pl.text(y=index, x=data, s=' %d (%0.02f%%)' % (data, (data/total_versions)*100))
pl

Now let's break down the number of versions per policy:

In [ ]:
from IPython.display import display, HTML
from pretty_html_table import build_table

ver = pd[['PolicyName', 'Version']].groupby('PolicyName').max()
ver.rename(columns={'Version': 'Versions', 'PolicyName': 'Policy'}, inplace=True)
ver.sort_values('Versions', ascending=False, inplace=True)
ver['Policy'] = ver.index
ver['Rank'] = range(1, ver.shape[0]+1)
ver_html = build_table(ver.head(19)[['Rank', 'Policy', 'Versions']], 'blue_light')
print(ver_html)
display(HTML(ver_html))

In [ ]:
ver.hist(column='Versions', bins=50)

In [ ]:
f"{ver[ver['Versions'] <= 5].shape[0] * 100 / ver.shape[0]}%% of policies have 5 versions or less."